# Azure FOCUS Export — Single-Notebook Pipeline

This notebook is fully self-contained: all code needed to create, run, and monitor Azure Cost Management FOCUS exports is included below.

## Authentication options

- `user` (recommended for interactive runs): uses your signed-in Azure user context (`az login`) and falls back to browser sign-in.
- `managed_identity`: for Fabric/hosted identities.
- `service_principal`: app auth with client secret (optional).
- `default`: `DefaultAzureCredential` chain.

If you use `user`, app registration secrets are **not required**.

## License Notice

This notebook and project are licensed under **AGPL-3.0-only**.

If you modify this notebook or code and share it (or run modified versions for users over a network), you must make the corresponding source code available under AGPL-3.0-only.

- Full license text: `LICENSE`
- SPDX identifier: `AGPL-3.0-only`

## Prerequisites and Permissions

Before running this notebook:

1. You have Azure RBAC on the **billing scope** (`subscription` or `billing_account`):
   - `Cost Management Contributor` (required to create/modify exports)
2. You have RBAC on the **storage account**:
   - `Storage Blob Data Contributor` (required to write export output)
3. Your storage account/container exist and match the values in the config cell.

## Recommended auth mode

For interactive usage, keep `AuthConfig(method="user")`.
- Primary path: signed-in Azure CLI user context (`az login`)
- Fallback path: interactive browser sign-in prompt
- No app secret is required for this mode.

In [ ]:
# SPDX-License-Identifier: AGPL-3.0-only
%pip install -q azure-identity requests tenacity rich

In [ ]:
# SPDX-License-Identifier: AGPL-3.0-only
import logging
import time
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from typing import Any, Dict, List, Optional, Tuple

import requests
from azure.identity import AzureCliCredential, ClientSecretCredential, DefaultAzureCredential, InteractiveBrowserCredential, ManagedIdentityCredential
from tenacity import retry, retry_if_exception, stop_after_attempt, wait_exponential

AZURE_MANAGEMENT_SCOPE = "https://management.azure.com/.default"
API_VERSION = "2025-03-01"
BASE_URL = "https://management.azure.com"
RETRYABLE_STATUS_CODES = {408, 429, 500, 502, 503, 504}

@dataclass
class AuthConfig:
    method: str = "user"
    tenant_id: Optional[str] = None
    client_id: Optional[str] = None
    client_secret: Optional[str] = None

@dataclass
class ScopeConfig:
    type: str = "subscription"
    subscription_id: Optional[str] = None
    billing_account_id: Optional[str] = None

    @property
    def scope_uri(self) -> str:
        if self.type == "subscription":
            if not self.subscription_id:
                raise ValueError("scope.subscription_id is required for subscription scope")
            return f"/subscriptions/{self.subscription_id}"
        if self.type == "billing_account":
            if not self.billing_account_id:
                raise ValueError("scope.billing_account_id is required for billing_account scope")
            return f"/providers/Microsoft.Billing/billingAccounts/{self.billing_account_id}"
        raise ValueError(f"Unknown scope type: {self.type}")

@dataclass
class StorageConfig:
    subscription_id: str = ""
    resource_group: str = ""
    account_name: str = ""
    container: str = "cost-exports"
    root_folder: str = "focus"

    @property
    def resource_id(self) -> str:
        return (
            f"/subscriptions/{self.subscription_id}"
            f"/resourceGroups/{self.resource_group}"
            f"/providers/Microsoft.Storage/storageAccounts/{self.account_name}"
        )

@dataclass
class ExportConfig:
    history_months: int = 36
    export_name_prefix: str = "focus-export"
    granularity: str = "Daily"
    partition_data: bool = True
    format: str = "Parquet"
    compression: str = "snappy"
    overwrite: bool = True
    request_timeout_seconds: int = 90
    throttle_delay_seconds: int = 8

@dataclass
class AppConfig:
    auth: AuthConfig = field(default_factory=AuthConfig)
    scope: ScopeConfig = field(default_factory=ScopeConfig)
    storage: StorageConfig = field(default_factory=StorageConfig)
    export: ExportConfig = field(default_factory=ExportConfig)

def validate_config(config: AppConfig) -> None:
    if config.auth.method not in {"user", "managed_identity", "service_principal", "default"}:
        raise ValueError("auth.method must be one of: user, managed_identity, service_principal, default")
    if config.auth.method == "service_principal":
        if not config.auth.tenant_id or not config.auth.client_id or not config.auth.client_secret:
            raise ValueError("service_principal auth requires tenant_id, client_id, and client_secret")
    _ = config.scope.scope_uri
    if not config.storage.subscription_id or not config.storage.resource_group or not config.storage.account_name:
        raise ValueError("storage.subscription_id/resource_group/account_name are required")

class AzureAuthenticator:
    def __init__(self, auth_config: AuthConfig):
        self._config = auth_config

    def get_token(self) -> str:
        method = self._config.method
        if method == "service_principal":
            credential = ClientSecretCredential(tenant_id=self._config.tenant_id, client_id=self._config.client_id, client_secret=self._config.client_secret)
            return credential.get_token(AZURE_MANAGEMENT_SCOPE).token
        if method == "managed_identity":
            kwargs = {}
            if self._config.client_id:
                kwargs["client_id"] = self._config.client_id
            return ManagedIdentityCredential(**kwargs).get_token(AZURE_MANAGEMENT_SCOPE).token
        if method == "default":
            return DefaultAzureCredential().get_token(AZURE_MANAGEMENT_SCOPE).token
        try:
            return AzureCliCredential().get_token(AZURE_MANAGEMENT_SCOPE).token
        except Exception:
            return InteractiveBrowserCredential(tenant_id=self._config.tenant_id).get_token(AZURE_MANAGEMENT_SCOPE).token

    def get_headers(self) -> dict:
        return {"Authorization": f"Bearer {self.get_token()}", "Content-Type": "application/json", "Accept": "application/json"}

class ExportsApiError(Exception):
    def __init__(self, status_code: int, message: str):
        self.status_code = status_code
        super().__init__(f"API error ({status_code}): {message}")

def _is_retryable_exception(exception: Exception) -> bool:
    if isinstance(exception, (requests.ConnectionError, requests.Timeout)):
        return True
    if isinstance(exception, ExportsApiError):
        return exception.status_code in RETRYABLE_STATUS_CODES
    return False

class ExportsApiClient:
    def __init__(self, authenticator: AzureAuthenticator, config: AppConfig):
        self._auth = authenticator
        self._config = config
        self._session = requests.Session()
        self._request_timeout_seconds = max(30, self._config.export.request_timeout_seconds)

    def _url(self, export_name: Optional[str] = None) -> str:
        scope = self._config.scope.scope_uri
        base = f"{BASE_URL}{scope}/providers/Microsoft.CostManagement/exports"
        if export_name:
            base = f"{base}/{export_name}"
        return f"{base}?api-version={API_VERSION}"

    def _handle(self, response: requests.Response) -> dict:
        if response.status_code in (200, 201):
            return response.json() if response.content else {}
        try:
            message = response.json().get("error", {}).get("message", response.text)
        except Exception:
            message = response.text
        raise ExportsApiError(response.status_code, message)

    @retry(stop=stop_after_attempt(8), wait=wait_exponential(multiplier=2, min=5, max=180), retry=retry_if_exception(_is_retryable_exception), reraise=True)
    def create_export(self, export_name: str, timeframe: str, start_iso: str, end_iso: str, schedule_status: str, recurrence: Optional[str] = None, recurrence_from: Optional[str] = None, recurrence_to: Optional[str] = None) -> dict:
        body: Dict[str, Any] = {
            "identity": {"type": "SystemAssigned"},
            "location": "centralus",
            "properties": {
                "format": self._config.export.format,
                "compressionMode": self._config.export.compression,
                "dataOverwriteBehavior": "OverwritePreviousReport" if self._config.export.overwrite else "CreateNewReport",
                "partitionData": self._config.export.partition_data,
                "definition": {
                    "type": "FocusCost",
                    "timeframe": timeframe,
                    "dataSet": {"configuration": {"dataVersion": "2023-05-01"}, "granularity": self._config.export.granularity}
                },
                "deliveryInfo": {
                    "destination": {
                        "type": "AzureBlob",
                        "container": self._config.storage.container,
                        "rootFolderPath": self._config.storage.root_folder,
                        "resourceId": self._config.storage.resource_id
                    }
                },
                "schedule": {"status": schedule_status}
            }
        }
        if timeframe == "Custom":
            body["properties"]["definition"]["timePeriod"] = {"from": start_iso, "to": end_iso}
        if recurrence:
            body["properties"]["schedule"]["recurrence"] = recurrence
        if recurrence_from and recurrence_to:
            body["properties"]["schedule"]["recurrencePeriod"] = {"from": recurrence_from, "to": recurrence_to}

        response = self._session.put(self._url(export_name), json=body, headers=self._auth.get_headers(), timeout=self._request_timeout_seconds)
        return self._handle(response)

    @retry(stop=stop_after_attempt(8), wait=wait_exponential(multiplier=2, min=5, max=180), retry=retry_if_exception(_is_retryable_exception), reraise=True)
    def run_export(self, export_name: str) -> None:
        scope = self._config.scope.scope_uri
        url = f"{BASE_URL}{scope}/providers/Microsoft.CostManagement/exports/{export_name}/run?api-version={API_VERSION}"
        response = self._session.post(url, headers=self._auth.get_headers(), timeout=self._request_timeout_seconds)
        if response.status_code not in (200, 202):
            self._handle(response)

    @retry(stop=stop_after_attempt(8), wait=wait_exponential(multiplier=2, min=5, max=180), retry=retry_if_exception(_is_retryable_exception), reraise=True)
    def list_exports(self) -> List[dict]:
        response = self._session.get(self._url(), headers=self._auth.get_headers(), timeout=self._request_timeout_seconds)
        data = self._handle(response)
        return data.get("value", [])

def month_ranges(months: int) -> List[Tuple[str, str]]:
    values = []
    now = datetime.utcnow()
    for i in range(months):
        target = now.replace(day=1) - timedelta(days=1)
        for _ in range(i):
            target = target.replace(day=1) - timedelta(days=1)
        year, month = target.year, target.month
        start = datetime(year, month, 1)
        end = datetime(year + 1, 1, 1) - timedelta(days=1) if month == 12 else datetime(year, month + 1, 1) - timedelta(days=1)
        values.append((start.strftime("%Y-%m-%dT00:00:00Z"), end.strftime("%Y-%m-%dT00:00:00Z")))
    values.reverse()
    return values

def monthly_name(prefix: str, start_iso: str) -> str:
    return f"{prefix}-{start_iso[:7]}"

print("Core helpers loaded.")

In [ ]:
# SPDX-License-Identifier: AGPL-3.0-only
config = AppConfig(
    auth=AuthConfig(method="user", tenant_id=None),
    scope=ScopeConfig(type="subscription", subscription_id="<your-subscription-id>"),
    storage=StorageConfig(
        subscription_id="<storage-subscription-id>",
        resource_group="<storage-resource-group>",
        account_name="<storage-account-name>",
        container="cost-exports",
        root_folder="focus"
    ),
    export=ExportConfig(
        history_months=36,
        export_name_prefix="focus-export",
        format="Parquet",
        compression="snappy",
        request_timeout_seconds=90,
        throttle_delay_seconds=8,
    )
)
validate_config(config)
auth = AzureAuthenticator(config.auth)
api = ExportsApiClient(auth, config)
token = auth.get_token()
print(f"Token acquired (length: {len(token)})")

## Execution Order

Run cells in this order:

1. **Cell 1**: install dependencies
2. **Cell 2**: load all helper classes/functions
3. **Cell 3**: configure scope/storage/auth and validate token acquisition
4. **Cell 4**: dry-run monthly seed plan
5. **Cell 5**: optional production historical seed (uncomment to execute)
6. **Cell 6**: review monthly recurring export and list exports

## Safety notes

- Keep dry-run first to validate scope and naming.
- Production cells are commented to avoid accidental API execution.
- Re-runs are safe if you check existing export names before create/run.

In [ ]:
# SPDX-License-Identifier: AGPL-3.0-only
# Dry-run seed preview
ranges = month_ranges(config.export.history_months)
for start_iso, end_iso in ranges:
    print(f"Would create: {monthly_name(config.export.export_name_prefix, start_iso)} ({start_iso} -> {end_iso})")

In [ ]:
# SPDX-License-Identifier: AGPL-3.0-only
# Production seed run (uncomment to execute)
# existing = {item.get('name', '') for item in api.list_exports()}
# for start_iso, end_iso in ranges:
#     name = monthly_name(config.export.export_name_prefix, start_iso)
#     if name in existing:
#         print(f"Skipping existing: {name}")
#         continue
#     api.create_export(name, "Custom", start_iso, end_iso, "Inactive")
#     api.run_export(name)
#     print(f"Started export: {name}")
#     time.sleep(config.export.throttle_delay_seconds)

In [ ]:
# SPDX-License-Identifier: AGPL-3.0-only
# Recurring monthly export
now = datetime.utcnow()
start = datetime(now.year + 1, 1, 1) if now.month == 12 else datetime(now.year, now.month + 1, 1)
end = datetime(now.year + 10, 12, 31)
monthly_name_value = f"{config.export.export_name_prefix}-monthly"
print(f"Would create/update recurring export: {monthly_name_value}")
# api.create_export(
#     export_name=monthly_name_value,
#     timeframe="TheLastMonth",
#     start_iso="",
#     end_iso="",
#     schedule_status="Active",
#     recurrence="Monthly",
#     recurrence_from=start.strftime("%Y-%m-%dT00:00:00Z"),
#     recurrence_to=end.strftime("%Y-%m-%dT00:00:00Z")
# )

focus_exports = [item for item in api.list_exports() if item.get('name', '').startswith(config.export.export_name_prefix)]
print(f"Found {len(focus_exports)} FOCUS exports")
for item in focus_exports:
    print(f"- {item.get('name', 'unknown')}")